In [1]:
# Author: Elena González Prieto
# Last modified: Nov 17, 2025

# --- Standard Library ---
import os
import sys
sys.path.append('../../')

# --- Libraries ---
import autograd.numpy as np
from autograd import grad, hessian

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# --- Scikit-learn ---
import sklearn
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    mean_squared_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import (
    GridSearchCV,
    PredefinedSplit,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

# --- Local Modules ---
from utils import *

In [2]:
#Load the data 

data = np.load('../../data_splits_splot22f_1215.npz')
X_train = data['X_train']
y_train = data['y_train'][:, :1]
X_val = data['X_val']
y_val = data['y_val'][:, :1]
X_test = data['X_test']
y_test = data['y_test'][:, :1]


print("Number of class 0 ", len(y_train[y_train==0]), len(y_val[y_val==0]),  len(y_test[y_test==0]))
print("Number of class 1 ", len(y_train[y_train==1]),  len(y_val[y_val==1]),  len(y_test[y_test==1]))
print("Number of class 2 ", len(y_train[y_train==2]), len(y_val[y_val==2]), len(y_test[y_test==2]))
print("Number of class 3 ", len(y_train[y_train==3]),  len(y_val[y_val==3]), len(y_test[y_test==3]))

#Standard normalize the training data and use the mean and std to normalize the testing data

knn_scaler = StandardScaler().fit(X_train)
X_train = knn_scaler.transform(X_train)
X_val   = knn_scaler.transform(X_val)
X_test  = knn_scaler.transform(X_test)

print("Training set ", np.shape(X_train), np.shape(y_train))
print("Validation set ", np.shape(X_val), np.shape(y_val))
print("Testing set ", np.shape(X_test), np.shape(y_test))


Number of class 0  1539 330 330
Number of class 1  8133 1742 1743
Number of class 2  8827 1892 1891
Number of class 3  905 194 194
Training set  (19404, 5) (19404, 1)
Validation set  (4158, 5) (4158, 1)
Testing set  (4158, 5) (4158, 1)


In [3]:
# Create a split indicator array
split_index = np.concatenate([
    np.full(len(X_train), -1),  # All train samples get -1
    np.zeros(len(X_val))         # All val samples get 0
])

# Combine train and validation sets
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.vstack([y_train, y_val]).ravel()


# Create the predefined split
ps = PredefinedSplit(test_fold=split_index)

# Do a gridsearch using kNN
estimator_KNN = KNeighborsClassifier(algorithm='auto')

parameters_KNN = {
    'n_neighbors': [ 3, 5, 7, 9, 11, 15, 20, 25],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']}

grid_search_KNN = GridSearchCV(
    estimator   = estimator_KNN,
    param_grid  = parameters_KNN,
    scoring     = 'balanced_accuracy',
    cv          = ps)

grid_search_KNN.fit(X_train_val, y_train_val)


# Save performance 
model = grid_search_KNN.best_estimator_

# Generate predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred_test)*100
test_balanced_accuracy = balanced_accuracy_score(y_test, y_pred_test)*100

print("Test Accuracy: {:.3f}%".format(test_accuracy))
print("Test Balanced Accuracy: {:.3f}%".format(test_balanced_accuracy))
print(grid_search_KNN.best_params_)
# Save everything you'll need for plotting
np.savez('../results/knn_results.npz',
         y_pred_train=y_pred_train,
         y_pred_val=y_pred_val,
         y_pred_test=y_pred_test)


save_obj = {
    "model": grid_search_KNN.best_estimator_,
    "scaler": knn_scaler,
    "best_params": grid_search_KNN.best_params_,
    "best_score": grid_search_KNN.best_score_,
    "cv_results": grid_search_KNN.cv_results_,
    "test_accuracy": test_accuracy, 
    "test_balanced_accuracy": test_balanced_accuracy
}

joblib.dump(save_obj, "../best_models/knn_best_model.pkl")


Test Accuracy: 95.984%
Test Balanced Accuracy: 91.657%
{'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'distance'}


['../best_models/knn_best_model.pkl']

*Checking on the best-performing model*

In [3]:
model_name = '../best_models/knn_best_model.pkl'
checkpoint = joblib.load(model_name)
params = checkpoint["best_params"]
score = checkpoint["best_score"]
print(params, score)

{'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'distance'} 0.8982161750818914


In [ ]:
# Decision Boundary 

# unique_rows = unique_rows.T
import matplotlib.colors as mcolors
unique_rows = np.array([[1.0,1.0]])
for unique in unique_rows:
    Mass1 = unique[0]
    Mass2 = unique[1]
    # labels = ['log10(b[RSUN])', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    labels = ['log10(rp/(R1+R2))', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    N_knn_gridsearch_plotter(X_train_og, y_train, X_test_og ,y_test, best_model_knn, knn_scaler,feature_idx=(0, 1), fixed_values={2: Mass1, 3: Mass1/Mass2}, labels = labels)